In [1]:
# ===== 1. SETUP & CONSTANTS =====
import tensorflow as tf
import numpy as np
import pandas as pd
import cv2
import os, glob, json, time, gc
from datetime import datetime
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.optimizers import Adam
from scipy import stats as sp_stats

# Hyperparameters
IMG_SIZE = 384
BATCH_SIZE = 8
SEEDS = [42, 123, 2024]
OUTPUT_DIR = "/kaggle/working/multiseed_clean"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set global seed untuk reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("Setup selesai.")
print(f"IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, SEEDS={SEEDS}")

2026-06-09 09:01:51.303843: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780995711.481089      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780995711.536349      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780995711.973807      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780995711.973847      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780995711.973849      23 computation_placer.cc:177] computation placer alr

Setup selesai.
IMG_SIZE=384, BATCH_SIZE=8, SEEDS=[42, 123, 2024]


In [2]:
# ===== 2. DATA PIPELINE =====
DATA_ROOT = "/kaggle/input/datasets/lihuayang111265/flood-semantic-segmentation-dataset/dataset/"

train_images = sorted(glob.glob(os.path.join(DATA_ROOT, "train", "images", "*.png")))
train_labels = sorted(glob.glob(os.path.join(DATA_ROOT, "train", "labels", "*.png")))
val_images   = sorted(glob.glob(os.path.join(DATA_ROOT, "val", "images", "*.png")))
val_labels   = sorted(glob.glob(os.path.join(DATA_ROOT, "val", "labels", "*.png")))

print(f"Train: {len(train_images)} | Val: {len(val_images)}")

def load_pair(image_path, mask_path):
    img = tf.io.read_file(image_path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method='bilinear')
    img = tf.cast(img, tf.float32)
    
    mask = tf.io.read_file(mask_path)
    mask = tf.io.decode_image(mask, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, [IMG_SIZE, IMG_SIZE], method='nearest')
    mask = tf.cast(mask > 127, tf.float32)
    return img, mask

def augment_train(img, mask):
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    img = tf.image.rot90(img, k)
    mask = tf.image.rot90(mask, k)
    if tf.random.uniform(()) > 0.5:
        crop_frac = tf.random.uniform((), 0.70, 1.0)
        crop_size = tf.cast(tf.cast(IMG_SIZE, tf.float32) * crop_frac, tf.int32)
        combined = tf.concat([img, mask * 255.0], axis=-1)
        combined = tf.image.random_crop(combined, [crop_size, crop_size, 4])
        combined = tf.image.resize(combined, [IMG_SIZE, IMG_SIZE], method='bilinear')
        img = combined[..., :3]
        mask = tf.cast(combined[..., 3:4] > 127, tf.float32)
    img = tf.image.random_brightness(img, max_delta=12)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    img = preprocess_input(img)
    return img, mask

def preprocess_val(img, mask):
    img = preprocess_input(img)
    return img, mask

# Buat dataset dasar (akan di-rebuild per seed)
def build_train_ds_seeded(seed):
    return (tf.data.Dataset.from_tensor_slices((train_images, train_labels))
            .shuffle(len(train_images), seed=seed)
            .map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
            .map(augment_train, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((val_images, val_labels))
          .map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
          .map(preprocess_val, num_parallel_calls=tf.data.AUTOTUNE)
          .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

Train: 600 | Val: 63


I0000 00:00:1780995726.875902      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780995726.883022      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [3]:
# ===== 3. LOSS & METRICS =====
SMOOTH = 1e-6

def tversky_index(y_true, y_pred, alpha=0.3, beta=0.7):
    yt = K.flatten(y_true)
    yp = K.flatten(y_pred)
    tp = K.sum(yt * yp)
    fn = K.sum(yt * (1 - yp))
    fp = K.sum((1 - yt) * yp)
    return (tp + SMOOTH) / (tp + alpha * fp + beta * fn + SMOOTH)

def focal_tversky_loss(y_true, y_pred, alpha=0.3, beta=0.7, gamma=1.33):
    ti = tversky_index(y_true, y_pred, alpha, beta)
    return K.pow(1 - ti, gamma)

def dice_coef(y_true, y_pred):
    yt = K.flatten(y_true)
    yp = K.flatten(y_pred)
    inter = K.sum(yt * yp)
    return (2. * inter + SMOOTH) / (K.sum(yt) + K.sum(yp) + SMOOTH)

def iou_metric(y_true, y_pred):
    yt = K.flatten(y_true)
    yp = K.flatten(K.cast(y_pred > 0.5, tf.float32))
    inter = K.sum(yt * yp)
    union = K.sum(yt) + K.sum(yp) - inter
    return (inter + SMOOTH) / (union + SMOOTH)

In [4]:
# ===== 4. MODEL ARCHITECTURE =====
def attention_gate(x, g, inter_channels):
    theta_x = layers.Conv2D(inter_channels, 1, padding='same')(x)
    phi_g = layers.Conv2D(inter_channels, 1, padding='same')(g)
    if theta_x.shape[1] != phi_g.shape[1]:
        phi_g = layers.UpSampling2D(size=(theta_x.shape[1] // phi_g.shape[1],
                                           theta_x.shape[2] // phi_g.shape[2]))(phi_g)
    f = layers.Activation('relu')(layers.add([theta_x, phi_g]))
    psi = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(f)
    return layers.multiply([x, psi])

def decoder_block(x, skip, filters, use_attention=True):
    x = layers.UpSampling2D((2, 2), interpolation='bilinear')(x)
    if use_attention:
        skip = attention_gate(skip, x, filters // 2)
    x = layers.Concatenate()([x, skip])
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
    return x

def build_attention_unet_resnet50():
    base = ResNet50(input_shape=[IMG_SIZE, IMG_SIZE, 3],
                    include_top=False, weights='imagenet')
    base.trainable = False
    s1 = base.get_layer('conv1_relu').output
    s2 = base.get_layer('conv2_block3_out').output
    s3 = base.get_layer('conv3_block4_out').output
    s4 = base.get_layer('conv4_block6_out').output
    b  = base.get_layer('conv5_block3_out').output

    d4 = decoder_block(b, s4, 256)
    d3 = decoder_block(d4, s3, 128)
    d2 = decoder_block(d3, s2, 64)
    d1 = decoder_block(d2, s1, 32)
    d0 = layers.UpSampling2D((2, 2), interpolation='bilinear')(d1)
    d0 = layers.Conv2D(32, 3, padding='same', use_bias=False)(d0)
    d0 = layers.BatchNormalization()(d0)
    d0 = layers.Activation('relu')(d0)
    out = layers.Conv2D(1, 1, activation='sigmoid', name='mask_out')(d0)
    return Model(inputs=base.input, outputs=out, name='AttUNet_ResNet50')

In [5]:
# ===== 5. HELPER + CORE TRAINING FUNCTION =====
def set_global_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    import random
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def evaluate_with_per_image(mdl):
    preds_all, masks_all = [], []
    for imgs, msks in val_ds:
        preds_all.append(mdl.predict(imgs, verbose=0))
        masks_all.append(msks.numpy())
    preds_all = np.concatenate(preds_all, 0).squeeze()
    masks_all = np.concatenate(masks_all, 0).squeeze()

    best_iou, best_th = 0, 0.5
    for th in np.arange(0.25, 0.80, 0.05):
        pb = (preds_all > th).astype(np.uint8)
        iou = np.logical_and(pb, masks_all).sum() / (pb.sum() + masks_all.sum() - np.logical_and(pb, masks_all).sum() + 1e-7)
        if iou > best_iou:
            best_iou, best_th = iou, th

    f1_list = []
    for i in range(preds_all.shape[0]):
        pred_i = (preds_all[i] > best_th).astype(np.uint8)
        gt = masks_all[i].astype(np.uint8)
        tp = np.logical_and(pred_i == 1, gt == 1).sum()
        fp = np.logical_and(pred_i == 1, gt == 0).sum()
        fn = np.logical_and(pred_i == 0, gt == 1).sum()
        f1 = 2 * tp / (2 * tp + fp + fn + 1e-7)
        f1_list.append(f1)

    return {
        'threshold': float(best_th),
        'mean_f1': float(np.mean(f1_list)),
        'agg_iou': float(np.logical_and((preds_all > best_th), masks_all).sum() / 
                         (np.sum(preds_all > best_th) + np.sum(masks_all) - 
                          np.logical_and((preds_all > best_th), masks_all).sum() + 1e-7))
    }

def train_variant_d_one_seed(seed):
    seed_dir = os.path.join(OUTPUT_DIR, f'seed_{seed}')
    os.makedirs(seed_dir, exist_ok=True)
    metrics_path = os.path.join(seed_dir, 'metrics.json')

    if os.path.exists(metrics_path):
        print(f"[SKIP] Seed {seed} sudah selesai.")
        with open(metrics_path) as f:
            return json.load(f)

    set_global_seed(seed)
    tf.keras.backend.clear_session()

    train_ds = build_train_ds_seeded(seed)
    model = build_attention_unet_resnet50()
    model.compile(optimizer=Adam(1e-3), loss=focal_tversky_loss, metrics=[dice_coef, iou_metric])

    # Phase 1
    history1 = model.fit(
        train_ds, validation_data=val_ds, epochs=50, verbose=1,
        callbacks=[
            EarlyStopping('val_iou_metric', mode='max', patience=8, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau('val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
        ]
    )

    # Phase 2
    for layer in model.layers:
        layer.trainable = True
    model.optimizer.learning_rate.assign(1e-5)
    model.compile(optimizer=Adam(learning_rate=1e-5), loss=focal_tversky_loss, metrics=[dice_coef, iou_metric])

    history2 = model.fit(
        train_ds, validation_data=val_ds, epochs=50, verbose=1,
        callbacks=[
            EarlyStopping('val_iou_metric', mode='max', patience=12, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau('val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
        ]
    )

    ev = evaluate_with_per_image(model)
    model.save_weights(os.path.join(seed_dir, 'model.weights.h5'))

    summary = {
        'seed': seed,
        'mean_f1': ev['mean_f1'],
        'agg_iou': ev['agg_iou'],
        'threshold': ev['threshold'],
        'phase1_epochs': len(history1.history['loss']),
        'phase2_epochs': len(history2.history['loss'])
    }

    with open(metrics_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"[DONE] Seed {seed} | F1={ev['mean_f1']*100:.2f}% | IoU={ev['agg_iou']*100:.2f}%")
    return summary

In [6]:
# ===== 6. MULTI-SEED EXECUTION =====
print("\n" + "="*70)
print("MULAI MULTI-SEED TRAINING (Varian D)")
print("="*70)

results = []
for seed in SEEDS:
    print(f"\n>>> Training Seed {seed}")
    try:
        res = train_variant_d_one_seed(seed)
        results.append(res)
    except Exception as e:
        print(f"[ERROR] Seed {seed} gagal: {e}")
        gc.collect()

# Simpan ringkasan
with open(os.path.join(OUTPUT_DIR, 'all_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print("\n" + "="*70)
print("TRAINING SELESAI")
print("="*70)


MULAI MULTI-SEED TRAINING (Varian D)

>>> Training Seed 42
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Epoch 1/50


I0000 00:00:1780995749.486774      67 service.cc:152] XLA service 0x7f43200027a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780995749.486817      67 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780995749.486822      67 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780995752.595473      67 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-06-09 09:02:44.601606: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-09 09:02:44.895398: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-09 09:02:45.106842: E external/local_xl

75/75 ━━━━━━━━━━━━━━━━━━━━ 95s 583ms/step - dice_coef: 0.8272 - iou_metric: 0.7858 - loss: 0.0813 - val_dice_coef: 0.7773 - val_iou_metric: 0.6347 - val_loss: 0.1873 - learning_rate: 0.0010
Epoch 2/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 24s 323ms/step - dice_coef: 0.8915 - iou_metric: 0.8443 - loss: 0.0399 - val_dice_coef: 0.9037 - val_iou_metric: 0.8522 - val_loss: 0.0568 - learning_rate: 0.0010
Epoch 3/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 23s 308ms/step - dice_coef: 0.9127 - iou_metric: 0.8634 - loss: 0.0308 - val_dice_coef: 0.9438 - val_iou_metric: 0.9116 - val_loss: 0.0236 - learning_rate: 0.0010
Epoch 4/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 24s 312ms/step - dice_coef: 0.9180 - iou_metric: 0.8653 - loss: 0.0281 - val_dice_coef: 0.9409 - val_iou_metric: 0.8980 - val_loss: 0.0170 - learning_rate: 0.0010
Epoch 5/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 24s 320ms/step - dice_coef: 0.9257 - iou_metric: 0.8743 - loss: 0.0252 - val_dice_coef: 0.9518 - val_iou_metric: 0.9149 - val_loss: 0.0157 - learning_rate: 0.0010
Epoch 